# odds-lab, step 2: from probabilities to prices

**Question:** given a probability for each outcome, what odds should an operator offer, and how much does it expect to keep?

Step 1 produced probabilities. This notebook turns them into odds with a margin, and then asks the uncomfortable follow-up: the model is not perfect, so how big must the margin be before the prices stop being a gift to better-informed bettors?

| Step | Question |
|---|---|
| 1 | Can past results be turned into match probabilities? |
| **2 (this notebook)** | How do probabilities become odds with a margin? |
| 3 | What does the operator's profit depend on? |
| 4 | Web app and build log |

Run the cells top to bottom. It is self-contained: the first code cell repeats the step 1 functions.

In [ ]:
# --- Setup: step 1 functions, unchanged (explained in the step 1 notebook) ---
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import poisson
from sklearn.linear_model import LogisticRegression

REQUIRED = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]
ODDS = ["B365H", "B365D", "B365A"]


def load_matches(folder: str | Path) -> pd.DataFrame:
    """Read every CSV in `folder`, one file per season, into one table sorted by date."""
    files = sorted(Path(folder).glob("*.csv"))
    if not files:
        raise FileNotFoundError(f"No CSV files in {folder}")
    frames = []
    for f in files:
        df = pd.read_csv(f, encoding="latin-1")
        missing = [c for c in REQUIRED + ODDS if c not in df.columns]
        if missing:
            raise ValueError(f"{f.name} is missing columns {missing}")
        df = df[REQUIRED + ODDS].dropna(subset=REQUIRED)
        # football-data.co.uk mixes dd/mm/yy and dd/mm/yyyy across seasons
        df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, format="mixed")
        df["season"] = f.stem
        frames.append(df)
    out = pd.concat(frames, ignore_index=True).sort_values("Date", kind="stable")
    out = out.dropna(subset=ODDS).reset_index(drop=True)
    bad = ~out["FTR"].isin(["H", "D", "A"])
    if bad.any():
        raise ValueError(f"{bad.sum()} rows have a result other than H/D/A")
    return out


def make_synthetic(folder: str | Path, seasons: int = 6, teams: int = 20,
                   margin: float = 0.05, seed: int = 7) -> None:
    """Write `seasons` fake league seasons in football-data.co.uk format.

    Each team has a hidden strength that drifts between seasons. Goals are
    Poisson. The fake bookmaker knows the true probabilities up to a little
    noise and adds `margin` on top - so on this data the bookmaker should
    beat any model, which is a useful sanity check.
    """
    rng = np.random.default_rng(seed)
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    names = [f"Team{i:02d}" for i in range(teams)]
    strength = rng.normal(0, 0.35, teams)
    goals = np.arange(0, 11)
    for s in range(seasons):
        rows = []
        day = pd.Timestamp(2019 + s, 8, 10)
        fixtures = [(h, a) for h in range(teams) for a in range(teams) if h != a]
        rng.shuffle(fixtures)
        for i, (h, a) in enumerate(fixtures):
            lam_h = np.exp(0.30 + strength[h] - strength[a])   # 0.30 = home edge
            lam_a = np.exp(0.05 + strength[a] - strength[h])
            ph = _pois(goals, lam_h)
            pa = _pois(goals, lam_a)
            grid = np.outer(ph, pa)
            p = np.array([np.tril(grid, -1).sum(), np.trace(grid), np.triu(grid, 1).sum()])
            p = p / p.sum()
            gh, ga = rng.poisson(lam_h), rng.poisson(lam_a)
            noisy = p * np.exp(rng.normal(0, 0.03, 3))
            noisy = noisy / noisy.sum()
            odds = np.round(1 / (noisy * (1 + margin)), 2)
            rows.append({
                "Date": (day + pd.Timedelta(days=i // 10 * 7)).strftime("%d/%m/%Y"),
                "HomeTeam": names[h], "AwayTeam": names[a],
                "FTHG": gh, "FTAG": ga,
                "FTR": "H" if gh > ga else ("D" if gh == ga else "A"),
                "B365H": odds[0], "B365D": odds[1], "B365A": odds[2],
            })
        pd.DataFrame(rows).to_csv(folder / f"synthetic_{2019 + s}.csv", index=False)
        strength = 0.85 * strength + rng.normal(0, 0.12, teams)


def _pois(k, lam):
    return poisson.pmf(k, lam)


START = 1500.0


def add_elo(matches: pd.DataFrame, k: float = 20.0, home_adv: float = 60.0) -> pd.DataFrame:
    """Return a copy of `matches` (must be date-sorted) with pre-match
    elo_home, elo_away and elo_diff (home minus away, before home advantage)."""
    if not matches["Date"].is_monotonic_increasing:
        raise ValueError("matches must be sorted by date")
    rating: dict[str, float] = {}
    eh, ea = [], []
    for home, away, res in zip(matches["HomeTeam"], matches["AwayTeam"], matches["FTR"]):
        rh, ra = rating.get(home, START), rating.get(away, START)
        eh.append(rh)
        ea.append(ra)
        expected_home = 1.0 / (1.0 + 10 ** (-(rh + home_adv - ra) / 400.0))
        score_home = {"H": 1.0, "D": 0.5, "A": 0.0}[res]
        delta = k * (score_home - expected_home)
        rating[home], rating[away] = rh + delta, ra - delta
    out = matches.copy()
    out["elo_home"], out["elo_away"] = eh, ea
    out["elo_diff"] = out["elo_home"] - out["elo_away"]
    return out


CLASSES = ["H", "D", "A"]
ODDS = ["B365H", "B365D", "B365A"]


def implied_probs(odds: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """odds (n,3) -> (fair probabilities (n,3), overround (n,))."""
    raw = 1.0 / np.asarray(odds, dtype=float)
    total = raw.sum(axis=1)
    return raw / total[:, None], total - 1.0


def split_by_season(df: pd.DataFrame, test_seasons: int = 1):
    """Train on the past, test on the most recent season(s). Never shuffle:
    a random split would let the model train on matches played after the
    ones it is tested on."""
    seasons = list(dict.fromkeys(df["season"]))
    if len(seasons) <= test_seasons:
        raise ValueError("need more seasons than test_seasons")
    test = df["season"].isin(seasons[-test_seasons:])
    return df[~test], df[test]


def fit(train: pd.DataFrame) -> LogisticRegression:
    model = LogisticRegression(C=1e6, max_iter=1000)  # C large = no regularisation
    model.fit(train[["elo_diff"]].to_numpy() / 100.0, train["FTR"])
    return model


def predict(model: LogisticRegression, df: pd.DataFrame) -> np.ndarray:
    """(n,3) probabilities in H, D, A order."""
    p = model.predict_proba(df[["elo_diff"]].to_numpy() / 100.0)
    order = [list(model.classes_).index(c) for c in CLASSES]
    return p[:, order]


def one_hot(results: pd.Series) -> np.ndarray:
    return np.stack([(results == c).to_numpy(dtype=float) for c in CLASSES], axis=1)


def brier(p: np.ndarray, y: np.ndarray) -> float:
    """Mean squared error of the probabilities. 0 = perfect, lower is better.
    Always saying 1/3 each scores 0.667."""
    return float(((p - y) ** 2).sum(axis=1).mean())


def log_loss(p: np.ndarray, y: np.ndarray) -> float:
    """Punishes confident wrong answers harder than Brier. Lower is better.
    Always saying 1/3 each scores 1.099."""
    return float(-(y * np.log(np.clip(p, 1e-12, 1))).sum(axis=1).mean())


def calibration(p: np.ndarray, y: np.ndarray, bins: int = 5) -> pd.DataFrame:
    """Pool all three outcomes, bucket by predicted probability, and compare
    with how often those outcomes happened. Calibrated = the columns match."""
    flat_p, flat_y = p.ravel(), y.ravel()
    edges = np.quantile(flat_p, np.linspace(0, 1, bins + 1))
    idx = np.clip(np.searchsorted(edges, flat_p, side="right") - 1, 0, bins - 1)
    t = pd.DataFrame({"bin": idx, "predicted": flat_p, "observed": flat_y})
    return t.groupby("bin").agg(predicted=("predicted", "mean"),
                                observed=("observed", "mean"),
                                n=("observed", "size")).round(3)


def evaluate(train: pd.DataFrame, test: pd.DataFrame, model: LogisticRegression) -> pd.DataFrame:
    """Score three forecasters on the test set: base rates, our model, the bookmaker."""
    y = one_hot(test["FTR"])
    base = np.tile(one_hot(train["FTR"]).mean(axis=0), (len(test), 1))
    ours = predict(model, test)
    book, _ = implied_probs(test[ODDS].to_numpy())
    rows = [(name, brier(p, y), log_loss(p, y))
            for name, p in [("base rates", base), ("elo + logistic", ours), ("bookmaker", book)]]
    return pd.DataFrame(rows, columns=["forecaster", "brier", "log_loss"]).round(4)

## 0. Data and step 1 model

The notebook looks for real season files in the folders listed in `CANDIDATES`, in order. In Colab, files uploaded to the session (`data/real`) are deleted when the runtime resets, so the second entry points at Google Drive: put the CSVs in `MyDrive/odds-lab/data/real` once and they stay.

In [ ]:
CANDIDATES = [Path("data/real"), Path("/content/drive/MyDrive/odds-lab/data/real")]
SYNTH = Path("data/synthetic")

DATA = next((c for c in CANDIDATES if list(c.glob("*.csv"))), None)
IS_REAL = DATA is not None
if IS_REAL:
    print(f"Using REAL season files from {DATA}")
else:
    DATA = SYNTH
    if not list(SYNTH.glob("*.csv")):
        make_synthetic(SYNTH)
    print("No real CSVs found, so using SYNTHETIC seasons (not real football)")

matches = add_elo(load_matches(DATA))
train, test = split_by_season(matches)
model = fit(train)

p_model = predict(model, test)                       # our probabilities, (n, 3) in H, D, A order
p_book, over_book = implied_probs(test[ODDS].to_numpy())
y_test = one_hot(test["FTR"])
print(f"train {len(train)} | test {len(test)} matches (season {test['season'].iloc[0]})")
evaluate(train, test, model)

## 1. Fair odds, and why nobody offers them

Fair decimal odds are $1/p$. A bet at fair odds has zero expected profit for both sides, so an operator offering them earns nothing on average and still carries the risk.

The operator's expected profit on a 1-unit bet at odds $o$ on an outcome with true probability $q$ is

$$\text{edge} = 1 - q \cdot o$$

It keeps the stake (1) and pays out $o$ with probability $q$. At fair odds $o = 1/q$ the edge is exactly 0. To earn anything the operator must offer odds *shorter* than fair.

## 2. Two ways to add a margin

Both methods inflate the probabilities so they add up to $1 + m$ (the overround from step 1), then take odds $= 1/\text{inflated probability}$. They differ in *where* the margin is placed.

**Proportional.** Multiply every probability by $(1+m)$. Every outcome carries the same edge, $m/(1+m)$.

**Power.** Raise every probability to a power $k < 1$, with $k$ chosen so the total is $1+m$. Small probabilities are inflated proportionally more, so **longshots carry more margin than favourites**. Real bookmakers price roughly like this (the "favourite-longshot bias"): a longshot is where a mistaken probability hurts most, and where bettors are least price-sensitive.

In [ ]:
def _solve_power(p, target):
    """Find k per row so that sum_i p_i**k == target (bisection; the sum falls as k rises)."""
    p = np.atleast_2d(np.asarray(p, dtype=float))
    lo, hi = np.full(len(p), 0.01), np.full(len(p), 10.0)
    for _ in range(60):
        mid = (lo + hi) / 2
        too_big = (p ** mid[:, None]).sum(axis=1) > target
        lo, hi = np.where(too_big, mid, lo), np.where(too_big, hi, mid)
    return (lo + hi) / 2


def booked_probs(p, margin, method="proportional"):
    """Fair probabilities (n,3) -> inflated probabilities summing to 1 + margin."""
    p = np.atleast_2d(np.asarray(p, dtype=float))
    if margin < 0:
        raise ValueError("margin cannot be negative")
    if method == "proportional":
        return p * (1 + margin)
    if method == "power":
        return p ** _solve_power(p, 1 + margin)[:, None]
    raise ValueError(f"unknown method {method!r}")


def price(p, margin, method="proportional", tick=0.01):
    """Fair probabilities -> decimal odds offered to the customer.
    Odds are rounded DOWN to the tick, so rounding never gives margin away."""
    odds = 1.0 / booked_probs(p, margin, method)
    if tick:
        odds = np.floor(odds / tick + 1e-9) * tick
    return np.maximum(odds, 1.01)


def remove_margin(odds, method="proportional"):
    """The inverse: offered odds -> the fair probabilities behind them."""
    raw = 1.0 / np.atleast_2d(np.asarray(odds, dtype=float))
    if method == "proportional":
        return raw / raw.sum(axis=1, keepdims=True)
    if method == "power":
        return raw ** _solve_power(raw, 1.0)[:, None]
    raise ValueError(f"unknown method {method!r}")


def edge(odds, q):
    """Operator's expected profit per unit staked, if the true probability is q."""
    return 1.0 - np.asarray(q) * np.asarray(odds)

**One match, priced both ways.** The last column is the operator's edge on each outcome *if our probabilities are right*.

In [ ]:
MARGIN = 0.06          # 6% overround: try 0.02 or 0.10 and re-run

i = len(test) - 1      # last match of the test season
m_row = test.iloc[i]
p_one = p_model[[i]]

rows = []
for method in ["proportional", "power"]:
    o = price(p_one, MARGIN, method)[0]
    for j, name in enumerate(["home", "draw", "away"]):
        rows.append({"method": method, "outcome": name, "our prob": p_one[0, j],
                     "fair odds": 1 / p_one[0, j], "offered odds": o[j],
                     "edge": edge(o[j], p_one[0, j])})
print(f"{m_row['HomeTeam']} v {m_row['AwayTeam']}  |  margin {MARGIN:.0%}  |  "
      f"bookmaker's odds: {m_row[ODDS].to_numpy()}")
pd.DataFrame(rows).round(3)

In [ ]:
# where each method puts the margin, across the whole probability range
ps = np.linspace(0.03, 0.9, 60)
fig, ax = plt.subplots(figsize=(7, 4))
for method in ["proportional", "power"]:
    e = []
    for p1 in ps:
        rest = (1 - p1) / 2                       # a three-way market: p1 and two equal others
        o = price([[p1, rest, rest]], MARGIN, method, tick=None)[0, 0]
        e.append(edge(o, p1))
    ax.plot(ps, e, label=method)
ax.set_xlabel("true probability of the outcome  (longshot  ->  favourite)")
ax.set_ylabel("operator edge per unit staked")
ax.set_title(f"Where a {MARGIN:.0%} margin lands")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 3. Which method does the bookmaker's data support?

A test on the data rather than an assumption. Take the bookmaker's odds, remove the margin each way, and score the resulting probabilities against what actually happened, over every season. The method with the lower log loss is the better description of how the margin was put on.

(On the synthetic data the two come out almost identical and the difference is noise. The real seasons are the test that matters.)

In [ ]:
y_all = one_hot(matches["FTR"])
odds_all = matches[ODDS].to_numpy()
demargin = pd.DataFrame(
    [(mth, brier(remove_margin(odds_all, mth), y_all), log_loss(remove_margin(odds_all, mth), y_all))
     for mth in ["proportional", "power"]],
    columns=["margin removed by", "brier", "log_loss"]).round(5)
print("REAL data" if IS_REAL else "SYNTHETIC data")
print(f"{len(matches)} matches, average overround {implied_probs(odds_all)[1].mean():.2%}")
demargin

## 4. The catch: our probabilities are not the truth

Section 2 assumed our model is right. Step 1 showed the bookmaker forecasts better than we do. So suppose a sharp bettor knows the bookmaker's fair probability $r$ and we offer odds $o$ built from our own probability. Whenever

$$r \cdot o > 1$$

that bet has positive expected value **for the bettor**: we have mispriced the outcome by more than our margin covers. Call such a price *exposed*.

The margin is therefore doing two jobs: it is the profit on ordinary customers, and it is the cushion against our own model error. The plot shows how much cushion our Elo model needs.

In [ ]:
def exposure(p_ours, p_ref, margin, method="proportional"):
    """Share of offered prices a bettor holding p_ref could bet with positive
    expected value, and the bettor's average edge on those prices."""
    o = price(p_ours, margin, method)
    bettor_ev = p_ref * o - 1.0                 # the bettor's expected profit per unit
    exposed = bettor_ev > 0
    return exposed.mean(), (bettor_ev[exposed].mean() if exposed.any() else 0.0)

p_ref = remove_margin(test[ODDS].to_numpy(), "proportional")
margins = np.arange(0, 31) / 100
curve = pd.DataFrame(
    [(m, mth, *exposure(p_model, p_ref, m, mth)) for m in margins for mth in ["proportional", "power"]],
    columns=["margin", "method", "exposed share", "bettor edge when exposed"])

fig, ax = plt.subplots(figsize=(7, 4))
for mth, g in curve.groupby("method"):
    ax.plot(g["margin"], g["exposed share"], label=mth)
ax.axvline(over_book.mean(), color="k", ls="--", lw=1)
ax.text(over_book.mean() + 0.004, 0.42, "bookmaker's\nown margin", fontsize=9)
ax.set_xlabel("margin we charge"); ax.set_ylabel("share of our prices that are exposed")
ax.set_title("How much margin does our model need?")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

def margin_needed(limit, method):
    g = curve[(curve["method"] == method) & (curve["exposed share"] <= limit)]
    return f"{g['margin'].min():.0%}" if len(g) else ">30%"

print("REAL data" if IS_REAL else "SYNTHETIC data: shows the code works, not a finding")
near = margins[np.abs(margins - over_book.mean()).argmin()]
print(f"At roughly the bookmaker's own margin ({over_book.mean():.1%}):")
print(curve[curve["margin"] == near].round(3).to_string(index=False))
print("\nMargin needed to get the exposed share under 10% / 5%:")
for mth in ["proportional", "power"]:
    print(f"  {mth:13s} {margin_needed(0.10, mth)} / {margin_needed(0.05, mth)}")

# where are the exposed prices? split outcomes by how likely the bookmaker thinks they are
bucket = pd.cut(p_ref.ravel(), [0, 0.2, 0.4, 1.0], labels=["longshot (<20%)", "middle (20-40%)", "favourite (>40%)"])
where = pd.DataFrame({"outcome type": bucket, "n": 1})
for mth in ["proportional", "power"]:
    where[f"exposed, {mth}"] = (p_ref * price(p_model, near, mth) > 1).ravel()
print(f"\nExposed share by outcome type at a {near:.0%} margin:")
print(where.groupby("outcome type", observed=True).agg(
    n=("n", "sum"), proportional=("exposed, proportional", "mean"), power=("exposed, power", "mean")).round(3))

**Reading this.** At zero margin a large share of our prices is exposed, as it must be: wherever we rate an outcome lower than the bookmaker does, our fair odds are longer than theirs. The share falls as the margin grows, and the gap between "our model's required margin" and "the bookmaker's actual margin" is the price of having a weaker model. The table shows which kind of outcome the damage comes from. An operator closes that gap in three ways: a better model, limits on what sharp customers may stake, or moving prices when money arrives. Step 3 simulates those.

## 5. Checks

Each line guards against a specific pricing mistake. They use fixed numbers, so they do not depend on the data.

In [ ]:
P = np.array([[0.50, 0.30, 0.20], [0.80, 0.15, 0.05], [0.34, 0.33, 0.33]])

for mth in ["proportional", "power"]:
    b = booked_probs(P, 0.06, mth)
    assert np.allclose(b.sum(axis=1), 1.06, atol=1e-9)
    assert (np.argsort(b, axis=1) == np.argsort(P, axis=1)).all()
    assert np.allclose(1 / price(P, 0.0, mth, tick=None), P, atol=1e-9)
print("ok  both methods: booked probabilities sum to 1 + margin, keep the ranking, and margin 0 gives fair odds")

o = price(P, 0.06, "proportional", tick=None)
assert np.allclose(edge(o, P), 0.06 / 1.06)
print("ok  proportional: every outcome has edge m / (1 + m)")

e = edge(price(P, 0.06, "power", tick=None), P)
assert e[1, 2] > e[1, 0] and e[0, 2] > e[0, 0]
print("ok  power: the longshot carries more margin than the favourite")

for mth in ["proportional", "power"]:
    assert np.allclose(remove_margin(price(P, 0.06, mth, tick=None), mth), P, atol=1e-8)
print("ok  round trip: removing the margin recovers the original probabilities")

fine, rounded = price(P, 0.06, tick=None), price(P, 0.06, tick=0.01)
assert (rounded <= fine + 1e-12).all() and (fine - rounded < 0.01).all()
assert (1 / rounded).sum(axis=1).min() >= 1.06 - 1e-9
print("ok  rounding only ever shortens the odds, so it never gives margin away")

try:
    booked_probs(P, -0.01); raise AssertionError("negative margin accepted")
except ValueError:
    print("ok  a negative margin is rejected")

wrong = P[:, ::-1]                                   # a deliberately bad model
shares = [exposure(wrong, P, m)[0] for m in (0.0, 0.05, 0.20, 5.0)]
assert all(a >= b for a, b in zip(shares, shares[1:])) and shares[-1] < shares[0]
assert exposure(P, P, 0.0001)[0] == 0
print("ok  exposure falls as margin rises, and a perfect model is never exposed")

## 6. What this shows, and what comes next

- Odds are probabilities with a margin on top. If the probabilities are right, the operator keeps about $m/(1+m)$ of everything staked, whoever wins.
- Where the margin is placed is a choice. Loading it onto longshots protects the outcomes where a wrong probability is most expensive.
- The margin is also insurance against the operator's own model. The weaker the model, the more margin it needs before its prices stop being exploitable.

So the variables that drive an operator's profit are now on the table: **margin, model accuracy, and who is betting**. Step 3 puts numbers on them with a simulation: a mix of casual and sharp bettors, stake limits, and the resulting distribution of profit and worst-case loss.